In [1]:
from sklearn.model_selection import train_test_split
import pandas as pd

train_data = pd.read_csv('./data/train.csv') # Importing training data

new_train_data = pd.read_csv("./data/train_new.csv")

train_data = pd.concat((train_data, new_train_data), axis = 1, join = "inner")

train_data["P"] = train_data["P"].fillna(0)
train_data["O"] = train_data["O"].fillna(train_data["O"].median())

X = train_data.drop(["time", "Y1", "Y2"], axis = 1) # Losing the time and target columns

# Setting target variables
y1 = train_data["Y1"]

y2 = train_data["Y2"]

# Creating train, test split
X_train, X_val, y1_train, y1_val, y2_train, y2_val = train_test_split(X, y1, y2)

In [2]:
from sklearn.decomposition import PCA

# Applying PCA to training set
X_train = (X_train - X_train.mean(axis=0)) / X_train.std(axis=0)

pca = PCA(n_components= 3)

X_pca = pca.fit_transform(X_train)

X_train["PC1"] = X_pca[:,0]
X_train["PC2"] = X_pca[:,1]
X_train["PC3"] = X_pca[:,2]

In [3]:
from sklearn.decomposition import PCA

# Applying PCA to validation set
X_val = (X_val - X_val.mean(axis=0)) / X_val.std(axis=0)

pca = PCA(n_components= 3)

X_pca = pca.fit_transform(X_val)

X_val["PC1"] = X_pca[:,0]
X_val["PC2"] = X_pca[:,1]
X_val["PC3"] = X_pca[:,2]

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression

# Choosing Features
X1_train = X_train[["G", "M", "J", "C", "E", "H", "N", "PC1"]]
X1_val = X_val[["G", "M", "J", "C", "E", "H", "N", "PC1"]]

# Modelling
modelY1 = LinearRegression()

modelY1.fit(X1_train, y1_train)

# Predicting
y1_pred = modelY1.predict(X1_val)

# Choosing Features

X2_train = X_train[["A" ,"PC2", "K", "B", "D", "F", "I", "K", "L"]]
X2_val = X_val[["A" ,"PC2", "K", "B", "D", "F", "I", "K", "L"]]

# Modelling
modelY2 = RandomForestRegressor(n_estimators=250, n_jobs = 4)

modelY2.fit(X2_train, y2_train)

# Predicting
y2_pred = modelY2.predict(X2_val)


# OUTPUTS
print(f"Predicted score : {(r2_score(y2_pred, y2_val) + r2_score(y1_pred, y1_val))/2}")
print(f"Score Y1 : {r2_score(y1_pred, y1_val)}\nScore Y2 : {r2_score(y2_pred, y2_val)}")

Predicted score : 0.6657687507847819
Score Y1 : 0.6979856625357502
Score Y2 : 0.6335518390338136


In [6]:
from sklearn.decomposition import PCA

test_data = pd.read_csv('./data/test.csv') # Importing testing data
new_test_data = pd.read_csv('./data/test_new.csv') # Importing new testing data

test_data = pd.concat((test_data, new_test_data), axis = 1, join = "inner")

test_data["P"] = test_data["P"].fillna(0)
test_data["O"] = test_data["O"].fillna(test_data["O"].median())

X = test_data.drop(["time"], axis = 1) # Losing the time column

# Applying PCA to testing set
X = (X - X.mean(axis=0)) / X.std(axis=0)

pca = PCA(n_components= 3)

X_pca = pca.fit_transform(X)

X["PC1"] = X_pca[:,0]
X["PC2"] = X_pca[:,1]
X["PC3"] = X_pca[:,2]

X1 = X[["G", "M", "J", "C", "E", "H", "N", "PC1"]]

y1_pred = pd.DataFrame(modelY1.predict(X1), index = test_data.id, columns = ["Y1"])

X2 = X[["A" ,"PC2", "K", "B", "D", "F", "I", "K", "L"]]

y2_pred = pd.DataFrame(modelY2.predict(X2), index = test_data.id, columns = ["Y2"])

out = pd.concat((y1_pred, y2_pred), axis = 1)

out.to_csv("./data/predictions.csv") # Scores 0.699